# <font color='blue'>Data Science Academy</font>
# <font color='blue'>Processamento de Linguagem Natural com Transformers</font>

## <font color='blue'>Lab 4</font>
## <font color='blue'>Convertendo Áudio em Texto com Inteligência Artificial</font>

![DSA](images/Lab4.png)

In [ ]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python Usada Neste Jupyter Notebook:', python_version())

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

In [ ]:
#!pip install -q transformers==4.30.2
!pip install -q transformers

In [ ]:
#!pip install -q torch==2.0.1
!pip install -q torch

https://pypi.org/project/soundfile/

In [ ]:
!pip install -q soundfile

https://pypi.org/project/librosa/

In [ ]:
!pip install -q librosa

https://pypi.org/project/datasets/

In [ ]:
!pip install -qU datasets

In [ ]:
# Imports
import torch
import librosa
import transformers
from datasets import load_dataset
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Data Science Academy" --iversions

## Modelo Facebook Wav2Vec2

Este é um modelo básico pré-treinado e ajustado em 960 horas de Librispeech em áudio de fala amostrado de 16kHz. Ao usar o modelo, certifique-se de que sua entrada de fala também seja amostrada em 16Khz.

https://arxiv.org/abs/2006.11477

https://huggingface.co/facebook/wav2vec2-base-960h

In [ ]:
# Carrega o tokenizer e o modelo
tokenizador_1 = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
modelo_1 = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

https://huggingface.co/datasets/patrickvonplaten/librispeech_asr_dummy/tree/main

In [ ]:
# Carrega conjunto de dados fictício de arquivos de áudio
dados = load_dataset("patrickvonplaten/librispeech_asr_dummy", "clean", split = "validation")

In [ ]:
dados

In [ ]:
dados['file']

In [ ]:
dados['audio']

In [ ]:
# Aplica o tokenizador em 1 arquivo de áudio
input_values = tokenizador_1(dados[0]["audio"]["array"], 
                             return_tensors = "pt", 
                             padding = "longest", 
                             sampling_rate = 16_000).input_values 

In [ ]:
input_values

In [ ]:
# Retorna os logits (previsões do modelo)
logits = modelo_1(input_values).logits

In [ ]:
# Argmax com os maiores logits para extrair as previsões
predicted_ids = torch.argmax(logits, dim = -1)

In [ ]:
# Previsão de texto do arquivo de áudio
predicted_ids

In [ ]:
# Tokeniza a previsão
transcription = tokenizador_1.batch_decode(predicted_ids)

In [ ]:
transcription

## Trabalhando com Áudio em Português

https://huggingface.co/jonatasgrosman/wav2vec2-large-xlsr-53-portuguese

In [ ]:
# Modelo
MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-portuguese"

In [ ]:
# Carrega o tokenizer e o modelo
tokenizer_2 = Wav2Vec2Processor.from_pretrained(MODEL_ID)
modelo_2 = Wav2Vec2ForCTC.from_pretrained(MODEL_ID)

In [ ]:
# Idioma dos áudios
LANG_ID = "pt"

In [ ]:
# Número de amostras
SAMPLES = 10

In [ ]:
# hf_token = input("Por favor, insira seu personal access token da HF: ")

In [ ]:
# Carrega os arquivos de áudio em português do dataset common_voice
dados_teste = load_dataset("common_voice", LANG_ID, split = f"test[:{SAMPLES}]")
# dados_teste = load_dataset("common_voice", LANG_ID, split=f"test[:{SAMPLES}]", download_mode="force_redownload")
# dados_teste = load_dataset("common_voice", LANG_ID, split=f"test[:{SAMPLES}]", streaming=True)
# dados_teste = load_dataset("common_voice", LANG_ID, split = f"test[:{SAMPLES}]", streaming=True)
# Load the entire 'test' split using streaming mode
# dados_teste = load_dataset("common_voice", LANG_ID, split="test", streaming=True)
# # Take the first SAMPLES examples from the streaming dataset
# subset = list(dados_teste.take(SAMPLES))

In [ ]:
dados_teste

In [ ]:
dados_teste['path']

In [ ]:
# Função que processa os arquivos de áudio e carregando como arrays
def processa_audio(batch):
    
    # Extrai array e sampling_rate dos arquivos de áudio
    speech_array, sampling_rate = librosa.load(batch["path"], sr = 16_000)
    
    # Array do áudio
    batch["speech"] = speech_array
    
    # Texto (label)
    batch["sentence"] = batch["sentence"].upper()
    
    return batch

In [ ]:
# Aplica a função
dados_teste = dados_teste.map(processa_audio)

In [ ]:
dados_teste

In [ ]:
dados_teste["speech"]

In [ ]:
# Aplica o tokenizador
inputs = tokenizer_2(dados_teste["speech"], return_tensors = "pt", padding = True, sampling_rate = 16_000)

In [ ]:
# Extrai os logits (previsões do modelo)
with torch.no_grad():
    logits = modelo_2(inputs.input_values, attention_mask = inputs.attention_mask).logits

In [ ]:
# Previsões
predicted_ids = torch.argmax(logits, dim=-1)

In [ ]:
# Tokenizer das previsões
predicted_sentences = tokenizer_2.batch_decode(predicted_ids)

In [ ]:
# Imprime valor real e valor previsto pelo modelo
for i, predicted_sentence in enumerate(predicted_sentences):
    print("-" * 100)
    print("Valor Real:", dados_teste[i]["sentence"])
    print("Previsão (Texto Extraído do Áudio):", predicted_sentence)

# Fim